In [1]:
NUM_CLUSTS = 9  # 6 to 10

SMOOTH_WIDTH = 5  # Odd>1: 3,5,7,9,...
TRAIN_DOWNSEL = 23  # small values for code test; set to 1 to output all.
VALID_DOWNSEL = 47  #  "
USE_PREPROC = True  # Read in saved meta and features frames

USE_LR1 = True
LR1_C = 0.10  # smaller --> fewer non-zero coeff.s
USE_LR2 = False
LR2_C = 0.9

LR_BLUR = 0.0

above_dir = "/kaggle/input/hms-harmful-brain-activity-classification/"
above_dir_preproc = "/kaggle/working/"

SHOW_EDA_PLOTS = False

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import pyarrow
import pyarrow.parquet as pq
import pyarrow.dataset as pads

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression



In [2]:
GLOBAL_SEED = 0
np.random.seed(GLOBAL_SEED)



In [3]:
HBA_number = 6
HBA_names = ["seizure", "lpd", "gpd", "lrda", "grda", "other"]
HBA_expert_names = ["Seizure", "LPD", "GPD", "LRDA", "GRDA", "Other"]
iHBA_of_expert = {"Seizure": 0, "LPD": 1, "GPD": 2, "LRDA": 3, "GRDA": 4, "Other": 5}
HBA_votes = [
    "seizure_vote",
    "lpd_vote",
    "gpd_vote",
    "lrda_vote",
    "grda_vote",
    "other_vote",
]
HBA_probs = [
    "seizure_prob",
    "lpd_prob",
    "gpd_prob",
    "lrda_prob",
    "grda_prob",
    "other_prob",
]
the4chains = ["LL", "RL", "LP", "RP"]

np.set_printoptions(precision=6, suppress=True)




In [4]:
def kld_score(solution, submission, eps=1e-15):
    """
    Calculate the average KL divergence score.
    Assumes columns are the probability columns (no row id).

    Stability: clip to avoid log(0) / division issues causing NaNs during validation.
    """
    sumsum = 0.0
    for prob_col in solution.columns.values:
        p = np.clip(solution[prob_col].to_numpy(dtype=float), eps, 1.0)
        q = np.clip(submission[prob_col].to_numpy(dtype=float), eps, 1.0)
        sumsum += np.nansum(-1.0 * p * np.log(q / p))
    return sumsum / (len(solution))




In [5]:
def read_hms_meta():
    """
    Read in the train.csv and test.csv files.
    Add total_vote, _prob columns, and vote entropy to train_meta.
    Add extra cols to test to allow the same processing as train:
        eeg[spectro]_sub_id, eeg[spectro]_label_offset_seconds, label_id
    """

    test_meta = pd.read_csv(above_dir + "test.csv")
    test_meta_len = len(test_meta)
    print("Test has length", test_meta_len)
    test_meta["eeg_sub_id"] = 0
    test_meta["eeg_label_offset_seconds"] = 0.0
    test_meta["spectrogram_sub_id"] = 0
    test_meta["spectrogram_label_offset_seconds"] = 0.0
    test_meta["label_id"] = test_meta.eeg_id

    train_meta = pd.read_csv(above_dir + "train.csv")
    train_meta_len = len(train_meta)
    print("Train has length", train_meta_len, " with:")

    train_meta["total_vote"] = (
        train_meta["seizure_vote"]
        + train_meta["lpd_vote"]
        + train_meta["gpd_vote"]
        + train_meta["lrda_vote"]
        + train_meta["grda_vote"]
        + train_meta["other_vote"]
    )
    train_meta["max_vote"] = np.max(
        np.array(
            [
                train_meta["seizure_vote"],
                train_meta["lpd_vote"],
                train_meta["gpd_vote"],
                train_meta["lrda_vote"],
                train_meta["grda_vote"],
                train_meta["other_vote"],
            ]
        ),
        axis=0,
    )

    for this_col in [
        "label_id",
        "eeg_id",
        "spectrogram_id",
        "patient_id",
        "total_vote",
    ]:
        print(
            "   ", len(train_meta[this_col].unique()), "unique " + this_col + " values."
        )

    if SHOW_EDA_PLOTS:
        print("\nHistogram of the total votes")
        plt.figure(figsize=(6, 3))
        plt.hist(train_meta["total_vote"], bins=55, log=True)
        plt.title("Histogram of Total Votes")
        plt.show()

    for col_pre in HBA_names:
        train_meta[col_pre + "_prob"] = (
            train_meta[col_pre + "_vote"] / train_meta["total_vote"]
        )

    print("Calculating voting entropy values ...")

    def calc_entropy(row):
        the_probs = np.clip(row[HBA_probs].values.astype(float), 1.0e-8, 1.0)
        return np.nansum(the_probs * -1 * np.log(the_probs))

    train_meta["entropy"] = train_meta.apply(calc_entropy, axis=1)

    return train_meta, test_meta




In [6]:
def prob_prob_scatter(name1, name2, probs2plot, clust_ids, iclust_order=[0]):
    """
    Make a prob1 vs prob2 scatter plot.
    - name1, name2 are 2 of the 6 HBA expert_consensus labels.
    - probs2plot is 6-column dataframe, e.g., train_meta[HBA_probs]
    - clust_ids is an array of cluster id integers, e.g., train_meta["clust_id"]
    Include an x at the cluster centers in the chosen axes.
    Use sqrt scaling to emphasize lower values.
    External: HBA_probs, iHBA_of_expert[ ], clust_centers
    """
    hba_clrs = [
        "orange",
        "blue",
        "red",
        "black",
        "green",
        "purple",
        "green",
        "red",
        "blue",
        "orange",
    ]  # up to 10 clusters
    if len(iclust_order) > 2:
        kmclrs = hba_clrs.copy()
        for iord, iclust in enumerate(iclust_order):
            kmclrs[iclust] = hba_clrs[iord]
    else:
        kmclrs = hba_clrs.copy()

    clstclrs = []
    for ilab in clust_ids:
        clstclrs.append(kmclrs[ilab])

    ixax = iHBA_of_expert[name1]
    iyax = iHBA_of_expert[name2]
    lenprob = len(probs2plot)
    plt.figure(figsize=(5, 5))
    plt.scatter(
        np.sqrt(probs2plot[HBA_probs[ixax]]) + 0.04 * (np.random.rand(lenprob) - 0.5),
        np.sqrt(probs2plot[HBA_probs[iyax]]) + 0.04 * (np.random.rand(lenprob) - 0.5),
        s=3,
        c=clstclrs,
        alpha=0.02,
    )
    for iclust in range(0, len(clust_centers)):
        plt.plot(
            np.sqrt([clust_centers[iclust, ixax]]),
            np.sqrt([clust_centers[iclust, iyax]]),
            c=kmclrs[iclust],
            marker="x",
            markersize=15,
        )
    plt.xlabel("sqrt( " + name1 + " )")
    plt.ylabel("sqrt( " + name2 + " )")
    plt.show()
    return kmclrs




In [7]:
def assemble_features(meta_frame, traintest="train", smooth_width=5, SHOW_PLOT=True):
    """
    Create a dataframe of spectrogram features from the meta_frame rows.
    Will include clust_id (i.e, the y) if it is in the input meta_frame.

    Bugfix (submission-unblocking): use NUM_CLUSTS for the plot title (num_clusts was not defined
    inside this function and could raise NameError when SHOW_PLOT=True).
    """
    freqs = np.array(range(100)) * 0.19525 + 0.59
    spect_trend = 150.0 / (1.0**2.3 + freqs ** (2.3))
    freqs[0] = 0.0  # separate the (unsmoothed) first bin from others
    freqs4 = np.array(4 * list(freqs))
    spect_trend4 = np.array(4 * list(spect_trend))

    if SHOW_PLOT:
        plt.figure(figsize=(10, 8))

    feats_frame = []
    last_spectro_id_str = "starting"

    print_every_nth = max([100, 100 * int(0.5 + len(meta_frame.index) / (100.0 * 15))])

    for irow in meta_frame.index:
        this_row = meta_frame.loc[irow]
        spectro_id_str = str(int(this_row.spectrogram_id))  # make sure int

        if spectro_id_str != last_spectro_id_str:
            if traintest != "test":
                spectro_file = (
                    above_dir + "train_spectrograms/" + spectro_id_str + ".parquet"
                )
            else:
                spectro_file = (
                    above_dir + "test_spectrograms/" + spectro_id_str + ".parquet"
                )
            pads_spectro = pads.dataset(spectro_file)
            this_spectro = pads_spectro.to_table().to_pandas()

        last_spectro_id_str = spectro_id_str

        loc_offset = int(this_row.spectrogram_label_offset_seconds / 2)

        middle4s = (
            this_spectro.iloc[loc_offset + 148, 1:]
            + this_spectro.iloc[loc_offset + 149, 1:]
            + this_spectro.iloc[loc_offset + 150, 1:]
            + this_spectro.iloc[loc_offset + 151, 1:]
        ) / (4.0 * spect_trend4)
        middle4s = np.clip(middle4s, 0.001, 1000.0)
        middle4s = middle4s.replace([np.nan, -np.inf, np.inf], 0.001)

        ratio4s = (
            this_spectro.iloc[loc_offset + 148, 1:]
            + this_spectro.iloc[loc_offset + 149, 1:]
            + this_spectro.iloc[loc_offset + 150, 1:]
            + this_spectro.iloc[loc_offset + 151, 1:]
        ) / (
            this_spectro.iloc[loc_offset + 149 - 56, 1:]
            + this_spectro.iloc[loc_offset + 149 - 40, 1:]
            + this_spectro.iloc[loc_offset + 149 - 24, 1:]
            + this_spectro.iloc[loc_offset + 149 + 56, 1:]
            + this_spectro.iloc[loc_offset + 149 + 40, 1:]
            + this_spectro.iloc[loc_offset + 149 + 24, 1:]
        )
        ratio4s = np.clip((6.0 / 4.0) * ratio4s, 0.01, 100.0)
        ratio4s = ratio4s.replace([np.nan, -np.inf, np.inf], 1.0)

        ratio4spre = np.log10(ratio4s)
        spect_mean = np.mean(middle4s)
        spect_median = np.median(middle4s)

        the4means = []
        the4medians = []
        for ispec in range(4):
            ibeg = ([0, 100, 200, 300])[ispec]
            iend = ibeg + 100
            the4means.append(np.mean(middle4s[ibeg:iend]))
            the4medians.append(np.median(middle4s[ibeg:iend]))

        middle4spre = np.log10(middle4s / spect_mean)

        middle4s = middle4spre.rolling(
            smooth_width, min_periods=smooth_width, center=True, closed=None
        ).mean()
        ratio4s = ratio4spre.rolling(
            smooth_width, min_periods=smooth_width, center=True, closed=None
        ).mean()

        for ioff in range(0, 400, 100):
            for ibin in range(int((smooth_width - 1) / 2)):  # assumes width is odd
                middle4s[ibin + ioff] = middle4spre[ibin + ioff]
                ratio4s[ibin + ioff] = ratio4spre[ibin + ioff]

        baseinds = np.insert(
            np.arange(int((smooth_width - 1) / 2), 100, smooth_width), 0, 0
        )
        select_inds = np.concatenate(
            (baseinds, 100 + baseinds, 200 + baseinds, 300 + baseinds)
        )

        freqs4ds = freqs4[select_inds]
        middle4sds = middle4s[select_inds]
        ratio4sds = ratio4s[select_inds]

        middle_feats = middle4sds.to_frame().T
        ratio_feats = ratio4sds.to_frame().T
        oldcols = ratio_feats.columns
        ratio_feats.columns = ["r" + c for c in oldcols]

        these_feats = pd.concat([middle_feats, ratio_feats], axis=1)

        spect_mean = np.log10(spect_mean)
        spect_median = np.log10(spect_median)
        the4means = np.log10(the4means)
        the4medians = np.log10(the4medians)
        these_feats["Mean"] = spect_mean
        these_feats["Median"] = spect_median
        for ispec in range(4):
            these_feats[the4chains[ispec] + "mean"] = the4means[ispec]
            these_feats[the4chains[ispec] + "median"] = the4medians[ispec]

        if "clust_id" in meta_frame.columns:
            these_feats["clust_id"] = this_row.clust_id

        if len(feats_frame) == 0:
            feats_frame = these_feats.copy()
        else:
            feats_frame = pd.concat([feats_frame, these_feats])

        if len(feats_frame) % print_every_nth == 0:
            print("... {} done...".format(len(feats_frame)))

        if SHOW_PLOT:
            this_clr = "blue"
            title_start = "Middle-8s Feature Values of " + traintest
            plt.title(title_start + " (smooth={})".format(smooth_width))
            if "clust_id" in meta_frame.columns:
                this_clust = this_row.clust_id
                this_clr = kmclrs[this_clust]
                plt.title(
                    title_start
                    + " (color-coded by the "
                    + "{} clusters, smooth={})".format(NUM_CLUSTS, smooth_width)
                )
            the_alpha = np.clip(0.05 * 350 / len(meta_frame), 0.003, 0.5)
            plt.plot(
                freqs4ds + smooth_width * 0.15 * (np.random.rand(len(freqs4ds)) - 0.5),
                middle4sds,
                ".",
                c=this_clr,
                markersize=10,
                alpha=the_alpha,
            )
            plt.plot(
                -1.2 + smooth_width * 0.15 * (np.random.rand(4) - 0.5),
                the4means + (0.0 * (np.random.rand(4) - 0.5)),
                ".",
                c=this_clr,
                markersize=10,
                alpha=the_alpha,
            )
            plt.ylim(-1.5, 1.5)
            plt.xlabel(
                "<-- Means are band < 0" + 20 * " " + "Frequency Bands (Hz)" + 50 * " "
            )
            plt.ylabel(
                "Amplitude (/ref-spectrum, /mean, and smoothed n={})".format(
                    smooth_width
                )
            )

    if SHOW_PLOT:
        plt.savefig("middle8s_" + traintest + "_features.png")
        plt.show()

    return feats_frame.reset_index().drop(columns=["index"])




In [8]:
def find_best_tamed_kl(solution_probs, pred_ids, centers, eps=1e-15):
    """
    Adjust the taming fraction for each cluster center to optimize KL.

    Bugfix (submission-unblocking): previously relied on implicit globals and mixed *_prob vs *_vote.
    Keep identical taming search logic, but pass inputs explicitly and score against HBA_votes.
    """
    mean_all_probs = np.array(
        [0.208319, 0.132120, 0.128532, 0.138913, 0.179294, 0.212822]
    )
    mean_all_probs = np.clip(mean_all_probs, 1e-6, 1.0)
    mean_all_probs = mean_all_probs / mean_all_probs.sum()

    tamed_fracs = 0.0 * np.ones(NUM_CLUSTS)
    tamed_centers = centers.copy()
    for iclust in range(NUM_CLUSTS):
        tamed_centers[iclust, :] = mean_all_probs

    best_fracs = tamed_fracs.copy()
    best_centers = tamed_centers.copy()

    submission_probs = solution_probs.copy()

    for iclust in range(NUM_CLUSTS):
        last_kl = 10.0
        for this_frac in np.arange(0.03, 1.00, 0.05):  # 0.03--0.98
            tamed_fracs[iclust] = this_frac
            this_cent = (
                tamed_fracs[iclust] * centers[iclust, :]
                + (1.0 - tamed_fracs[iclust]) * mean_all_probs
            )
            tamed_centers[iclust, :] = this_cent

            for iprob in range(HBA_number):
                this_col_probs = tamed_centers[:, iprob]
                submission_probs[HBA_votes[iprob]] = this_col_probs[pred_ids]

            this_kl = kld_score(solution_probs, submission_probs, eps=eps)
            if this_kl < last_kl:
                best_fracs = tamed_fracs.copy()
                best_centers = tamed_centers.copy()
                last_kl = this_kl
            else:
                tamed_fracs[iclust] = best_fracs[iclust]
                tamed_centers[iclust, :] = best_centers[iclust, :]
                break
    return best_fracs, best_centers




In [9]:
def normalize_submission_probas(df, vote_cols, eps=1e-6):
    """Safety: ensure finite, strictly positive, row-normalized probabilities."""
    vals = df[vote_cols].to_numpy(dtype=np.float64)
    vals = np.nan_to_num(
        vals,
        nan=1.0 / len(vote_cols),
        posinf=1.0 / len(vote_cols),
        neginf=1.0 / len(vote_cols),
    )
    vals = np.clip(vals, eps, 1.0)
    row_sums = vals.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0.0, 1.0, row_sums)
    vals = vals / row_sums
    df[vote_cols] = vals
    return df




In [10]:
def align_features_for_model(X_df, feature_names):
    """
    Score-improving robustness: ensure test/valid have the exact RF feature set.
    If any columns are missing (can happen with cached CSVs across versions), add them as zeros.
    Also drop any unexpected extras so sklearn doesn't error or (worse) misalign.
    """
    X_aligned = X_df.reindex(columns=list(feature_names), fill_value=0.0)
    return X_aligned




In [11]:
def align_features_for_lr(X_df, lr_feature_names):
    """
    Submission-unblocking + score stability: LR requires exact columns/order seen in fit().
    We align by name and fill missing with 0.0 to avoid shape mismatch errors.
    """
    return X_df.reindex(columns=lr_feature_names, fill_value=0.0)




In [12]:
def centers_from_membership(membership_proba, centers):
    """
    Score-improving, metric-aligned change (KL): replace hard cluster id -> center lookup
    with soft membership weighting. This preserves the same RF model and centers, just uses
    predict_proba information to produce smoother probabilities.
    """
    # membership_proba: (n_samples, n_clusters)
    # centers: (n_clusters, n_classes)
    return membership_proba @ centers




In [13]:
train_meta, test_meta = read_hms_meta()



Test has length 9850
Train has length 96950  with:
    96950 unique label_id values.
    15396 unique eeg_id values.
    10024 unique spectrogram_id values.
    1910 unique patient_id values.
    25 unique total_vote values.
Calculating voting entropy values ...


In [14]:
if SHOW_EDA_PLOTS:
    plt.figure(figsize=(5, 2))
    plt.hist(train_meta["spectrogram_sub_id"], bins=55, log=True)
    plt.title("Histogram of spectrogram_sub_id")
    plt.show()

    plt.figure(figsize=(5, 2))
    plt.hist(train_meta["eeg_sub_id"], bins=55, log=True)
    plt.title("Histogram of eeg_sub_id")
    plt.show()



In [15]:
num_clusts = NUM_CLUSTS  # 6 to 10
clust_rows_bool = train_meta.eeg_sub_id < 200



In [16]:
prob_vectors = train_meta.loc[clust_rows_bool, HBA_probs]

print("\nUsing {} HBA samples for clustering.".format(len(prob_vectors)))
print(
    "These include {} unique eeg_ids".format(
        train_meta.loc[clust_rows_bool, "eeg_id"].nunique()
    ),
    "and {} unique patient ids.".format(
        train_meta.loc[clust_rows_bool, "patient_id"].nunique()
    ),
)

if SHOW_EDA_PLOTS:
    plt.figure(figsize=(6, 3))
    plt.hist(train_meta.loc[clust_rows_bool, "total_vote"], bins=55, log=True)
    plt.title("Histogram of total_vote in HBA samples clustered")
    plt.show()

prob_array = np.array(prob_vectors)
kmeans = KMeans(
    n_clusters=num_clusts,
    init="k-means++",
    n_init=10,
    max_iter=300,
    random_state=GLOBAL_SEED,
)
kmeans.fit(prob_array)

clust_centers = kmeans.cluster_centers_

clust_centers = np.clip(clust_centers, 1e-12, 1.0)
clust_centers = clust_centers / clust_centers.sum(axis=1, keepdims=True)

print("cluster centers:")
print(clust_centers)

iclust_of_order = []
for icol in range(HBA_number):
    iclust_of_order.append(np.argmax(clust_centers[:, icol]))
clust_by_max = np.argsort(-1 * np.max(clust_centers, axis=1))
for iord in range(HBA_number, num_clusts):
    iclust_of_order.append(clust_by_max[iord])

kmnames = HBA_expert_names.copy()
for ihyb in range(1, (num_clusts - HBA_number) + 1):
    kmnames.append("Hybrid-" + str(ihyb))




Using 94385 HBA samples for clustering.
These include 15396 unique eeg_ids and 1910 unique patient ids.
cluster centers:
[[0.015832 0.028127 0.017942 0.033896 0.026772 0.87743 ]
 [0.972452 0.005737 0.004993 0.002097 0.001091 0.01363 ]
 [0.002496 0.003722 0.005318 0.022309 0.954715 0.011439]
 [0.049103 0.480226 0.052148 0.159937 0.020805 0.237781]
 [0.007077 0.024344 0.004139 0.890302 0.061458 0.012681]
 [0.075    0.046335 0.752753 0.004347 0.025558 0.096007]
 [0.0302   0.051824 0.010405 0.531111 0.044086 0.332374]
 [0.02551  0.917721 0.013777 0.011591 0.002075 0.029325]
 [0.013312 0.027391 0.077005 0.063424 0.438721 0.380147]]


In [17]:
train_meta["clust_id"] = kmeans.predict(np.array(train_meta[HBA_probs]))

all_probs = train_meta[HBA_probs]
all_ids = train_meta["clust_id"]

if SHOW_EDA_PLOTS:
    kmclrs = prob_prob_scatter("Seizure", "GPD", all_probs, all_ids, iclust_of_order)
    kmclrs = prob_prob_scatter("LPD", "GRDA", all_probs, all_ids, iclust_of_order)
    kmclrs = prob_prob_scatter("LRDA", "Other", all_probs, all_ids, iclust_of_order)
else:
    kmclrs = [
        "orange",
        "blue",
        "red",
        "black",
        "green",
        "purple",
        "green",
        "red",
        "blue",
        "orange",
    ]

clust_counts = train_meta.clust_id.value_counts()

iorder_of_clust = num_clusts * [-1]
for iord, iclust in enumerate(iclust_of_order):
    iorder_of_clust[iclust] = iord
    if SHOW_EDA_PLOTS:
        if iord == 0:
            print("The close-to-unit-vector cluster centers:")
        if iord == 6:
            print("The Hybrid cluster centers:")
        plt.figure(figsize=(5, 1))
        plt.bar(HBA_expert_names, clust_centers[iclust, :], color=kmclrs[iclust])
        plt.ylim(-0.01, 1.01)
        plt.title(
            kmnames[iord]
            + "  kmclust={} has {} samples".format(iclust, clust_counts[iclust]),
            size="medium",
        )
        if iord < 5:
            plt.xticks([])
        plt.show()



In [18]:
solution_train_probs = train_meta[HBA_probs].copy()
solution_train_probs.columns = HBA_votes

submission_train_probs = solution_train_probs.copy()
clust_ids = train_meta["clust_id"].to_numpy()

for iprob in range(HBA_number):
    this_col_probs = clust_centers[:, iprob]
    submission_train_probs[HBA_votes[iprob]] = this_col_probs[clust_ids]

print(
    "Score if HBA samples are correctly assigned cluster prob.s:",
    np.round(kld_score(solution_train_probs, submission_train_probs), 4),
)



Score if HBA samples are correctly assigned cluster prob.s: 0.2324


In [19]:
smooth_width = 9  # Here smooth_width is just for looking,

spectro_meta = train_meta[clust_rows_bool].copy()
every_nth = 97

freqs = np.array(range(100)) * 0.19525 + 0.59
downsel_freqs = np.insert(
    freqs[int((smooth_width - 1) / 2) : 100 : smooth_width], 0, freqs[0]
)

spect_trend = 150.0 / (1.0**2.3 + freqs ** (2.3))



In [20]:
if SHOW_EDA_PLOTS:
    if NUM_CLUSTS < 10:
        fig = plt.figure(figsize=(10, 10))
        gs = fig.add_gridspec(3, 3, hspace=0, wspace=0)
        (ax0, ax1, ax2), (ax3, ax4, ax5), (ax6, ax7, ax8) = gs.subplots(
            sharex="col", sharey="row"
        )
        pltaxs = [ax0, ax1, ax2, ax3, ax4, ax5, ax6, ax7, ax8]
    else:
        fig = plt.figure(figsize=(10, 13.5))
        gs = fig.add_gridspec(4, 3, hspace=0, wspace=0)
        (ax0, ax1, ax2), (ax3, ax4, ax5), (ax6, ax7, ax8), (ax9, ax10, ax11) = (
            gs.subplots(sharex="col", sharey="row")
        )
        pltaxs = [ax0, ax1, ax2, ax3, ax4, ax5, ax6, ax7, ax8, ax9, ax10, ax11]

    all_medians = []
    all_means = []
    all_clusts = []
    print(
        "Plotting {} x 4 processed spectra".format(int(len(spectro_meta) / every_nth))
    )
    print_every_nth = max(
        [100, 100 * int(0.5 + (len(spectro_meta) / every_nth) / (100.0 * 15))]
    )

    for ilocrow in range(0, len(spectro_meta), every_nth):
        this_row = spectro_meta.iloc[ilocrow]
        this_clust = this_row.clust_id
        spectro_id_str = str(this_row.spectrogram_id)
        spectro_file = above_dir + "train_spectrograms/" + spectro_id_str + ".parquet"
        pads_spectro = pads.dataset(spectro_file)
        this_spectro = pads_spectro.to_table().to_pandas()
        loc_offset = int(this_row.spectrogram_label_offset_seconds / 2)

        for ispec in range(4):
            ibeg = ([1, 101, 201, 301])[ispec]
            iend = ibeg + 100
            middle4s = (
                this_spectro.iloc[loc_offset + 148, ibeg:iend]
                + this_spectro.iloc[loc_offset + 149, ibeg:iend]
                + this_spectro.iloc[loc_offset + 150, ibeg:iend]
                + this_spectro.iloc[loc_offset + 151, ibeg:iend]
            ) / (4.0 * spect_trend)
            middle4s = np.clip(middle4s, 0.001, 1000.0)
            middle4s = middle4s.replace([np.nan, -np.inf, np.inf], 0.001)

            spect_mean = np.log10(np.mean(middle4s))
            all_means.append(spect_mean)
            spect_median = np.log10(np.median(middle4s))
            all_medians.append(spect_median)
            all_clusts.append(this_clust)

            middle4s = np.log10(middle4s)
            middle4spre = middle4s - spect_mean
            middle4s = middle4spre.rolling(
                smooth_width, min_periods=smooth_width, center=True, closed=None
            ).mean()
            for ibin in range(int((smooth_width - 1) / 2)):
                middle4s[ibin] = middle4spre[ibin]

            pltaxs[iorder_of_clust[this_clust]].plot(
                (freqs), middle4s, c=kmclrs[this_clust], lw=3, alpha=0.01
            )

        if (ilocrow / every_nth + 1) % print_every_nth == 0:
            print("... {} done...".format(int(ilocrow / every_nth) + 1))

    for iax in range(NUM_CLUSTS):
        pltaxs[iax].text(8.5, 0.8, kmnames[iax], fontsize=16)
        pltaxs[iax].plot([0.0, 20.0], [0.0, 0.0], c="black", lw=3, alpha=0.2)
        pltaxs[iax].plot(downsel_freqs, len(downsel_freqs) * [0.0], ".k")
        pltaxs[iax].set_ylim(-1.0, 1.0)
        pltaxs[iax].set_xlim(0.0, 20.5)
        if (iax == 0) or (iax == 3) or (iax == 6) or (iax == 9):
            pltaxs[iax].set_ylabel("log10[ Spectrum /reference /mean" + " & smoothed]")
        if (iax == 6) or (iax == 7) or (iax == 8):
            pltaxs[iax].set_xlabel("Frequency (Hz)")

    fig.suptitle(
        "Spectra of middle 8s (color-coded by the {} clusters, smooth={})".format(
            num_clusts, smooth_width
        )
    )
    plt.savefig("middle8s_spectra.png")
    plt.show()
else:
    all_means, all_medians, all_clusts = [], [], []



In [21]:
if SHOW_EDA_PLOTS and len(all_means) > 0:
    print("\nMedian of the Means(below): {:.4f}".format(np.median(all_means)))
    plt.figure(figsize=(6, 2))
    plt.hist(np.clip(all_means, -2.0, 2.0), bins=100)
    plt.xlim(-2.05, 2.05)
    plt.xlabel("Mean")
    plt.title("Histogram of the Means of the log(Spectra/Ref-spect)")
    plt.show()

    print("\nMedian of the Medians(below): {:.4f}".format(np.median(all_medians)))
    plt.figure(figsize=(6, 2))
    plt.hist(np.clip(all_medians, -2.0, 2.0), bins=100)
    plt.xlim(-2.05, 2.05)
    plt.xlabel("log10( Median )")
    plt.title("Histogram of the Medians of the log(Spectra/Ref-spect)")
    plt.show()

    mmclrs = []
    for ilab in all_clusts:
        mmclrs.append(kmclrs[ilab])
    plt.figure(figsize=(3, 3))
    plt.scatter(all_medians, all_means, s=2, c=mmclrs, alpha=0.3)
    plt.xlim(-1.0, 1)
    plt.ylim(-1.0, 1)
    plt.xlabel("Median")
    plt.ylabel("Mean")
    plt.show()



In [22]:
train_rows_bool = (
    (train_meta.eeg_sub_id < 33 + 1) & (train_meta.eeg_sub_id % 5 == 3)
) | ((train_meta.eeg_sub_id == 0) & ((train_meta.eeg_id % 23) % 8 > 1))
print("Number of Training rows:", sum(train_rows_bool))

valid_rows_bool = (
    (train_meta.eeg_sub_id < 44 + 1) & (train_meta.eeg_sub_id % 19 == 6)
) | ((train_meta.eeg_sub_id == 0) & ((train_meta.eeg_id % 23) % 8 < 2))
print("Number of Validation rows:", sum(valid_rows_bool))



Number of Training rows: 24573
Number of Validation rows: 8701


In [23]:
preproc_train_meta_path = os.path.join(above_dir_preproc, "Xy_train_meta_v47.csv")
preproc_train_feats_path = os.path.join(above_dir_preproc, "Xy_train_feats_v47.csv")

use_preproc_now = (
    USE_PREPROC
    and os.path.exists(preproc_train_meta_path)
    and os.path.exists(preproc_train_feats_path)
)
if USE_PREPROC and (not use_preproc_now):
    print(
        "Preprocessed train files not found at:",
        above_dir_preproc,
        " -> assembling features instead.",
    )

if use_preproc_now:
    Xy_train_meta = pd.read_csv(preproc_train_meta_path)
    Xy_train_feats = pd.read_csv(preproc_train_feats_path)
    Xy_train_meta["clust_id"] = kmeans.predict(np.array(Xy_train_meta[HBA_probs]))
    Xy_train_feats["clust_id"] = Xy_train_meta["clust_id"]
    SMOOTH_WIDTH = 5
else:
    Xy_train_meta = (train_meta[train_rows_bool])[::TRAIN_DOWNSEL].copy()
    Xy_train_meta = Xy_train_meta.reset_index().drop(columns=["index"])
    print("Number of samples used for training =", len(Xy_train_meta))

    Xy_train_feats = assemble_features(
        Xy_train_meta, traintest="train", smooth_width=SMOOTH_WIDTH, SHOW_PLOT=False
    )
    Xy_train_meta.to_csv(
        os.path.join(above_dir_preproc, "Xy_train_meta_v47.csv"),
        header=True,
        index=False,
        float_format="%.6f",
    )
    Xy_train_feats.to_csv(
        os.path.join(above_dir_preproc, "Xy_train_feats_v47.csv"),
        header=True,
        index=False,
        float_format="%.6f",
    )



Preprocessed train files not found at: /kaggle/working/  -> assembling features instead.
Number of samples used for training = 1069


/tmp/ipykernel_11/3100455626.py:91: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  middle4s[ibin + ioff] = middle4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:91: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  middle4s[ibin + ioff] = middle4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:92: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ratio4s[ibin + ioff] = ratio4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:92: Futu

... 100 done...


/tmp/ipykernel_11/3100455626.py:91: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  middle4s[ibin + ioff] = middle4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:91: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  middle4s[ibin + ioff] = middle4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:92: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ratio4s[ibin + ioff] = ratio4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:92: Futu

... 200 done...


...

/tmp/ipykernel_11/3100455626.py:91: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  middle4s[ibin + ioff] = middle4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:91: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  middle4s[ibin + ioff] = middle4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:92: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ratio4s[ibin + ioff] = ratio4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:92: Futu

... 900 done...


/tmp/ipykernel_11/3100455626.py:91: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  middle4s[ibin + ioff] = middle4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:91: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  middle4s[ibin + ioff] = middle4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:92: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ratio4s[ibin + ioff] = ratio4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:92: Futu

... 1000 done...


/tmp/ipykernel_11/3100455626.py:91: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  middle4s[ibin + ioff] = middle4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:91: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  middle4s[ibin + ioff] = middle4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:92: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ratio4s[ibin + ioff] = ratio4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:92: Futu

In [24]:
Xy_train_feats



,LL_0.59,LL_0.98,LL_1.95,LL_2.93,LL_3.91,LL_4.88,LL_5.86,LL_6.84,LL_7.81,LL_8.79,...,Median,LLmean,LLmedian,RLmean,RLmedian,LPmean,LPmedian,RPmean,RPmedian,clust_id
0,-2.049207,-1.572406,-0.864111,-0.596751,-0.595304,-0.397050,-0.289951,-0.323832,-0.532517,-0.627249,...,0.219908,0.194167,0.145224,0.802068,0.478890,0.240322,0.154471,0.686278,0.405229,1
1,-0.534020,-0.199268,0.123374,0.058622,-0.140026,0.014729,-0.082327,-0.155657,-0.151992,-0.128668,...,-0.293503,-0.303723,-0.335458,-0.322135,-0.319356,-0.258955,-0.258701,-0.210042,-0.249256,2
2,0.797090,0.793865,-0.333304,-0.776784,-0.865223,-0.757663,-0.771573,-0.899795,-0.883142,-0.531897,...,0.868360,1.420636,0.944018,1.492152,0.990260,1.257688,0.749170,1.185628,0.841712,1
3,0.869597,0.931231,0.338556,-0.226893,-0.416519,-0.115759,-0.098450,0.049774,0.013731,0.012901,...,0.371321,0.736186,0.591423,0.075993,0.029837,0.801126,0.599383,0.228044,0.153125,1
4,-0.516979,-0.404133,-0.581760,-0.627438,-0.506515,-0.708402,-0.634921,-0.629830,-0.547939,-0.465344,...,-0.504102,-0.530540,-0.801278,-0.444120,-0.645645,-0.262421,-0.475216,-0.200570,-0.350558,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1064,-0.797558,-0.213660,0.144428,0.262621,0.193039,0.078641,-0.179645,-0.301369,-0.250557,-0.188335,...,-0.118550,-0.092705,-0.178147,-0.099260,-0.164225,0.047449,-0.091898,0.005782,0.050080,1
1065,-1.596018,-0.950276,0.134876,0.271453,0.463766,0.382603,0.259663,0.203488,0.035840,0.056870,...,0.262629,0.491702,0.471334,0.364725,0.193939,0.297599,0.143992,0.264450,0.125008,5
1066,-0.081216,-0.045332,-0.425936,-0.587067,-0.397958,-0.122230,0.129723,0.427053,0.316714,0.115171,...,-0.692198,-0.652095,-0.828435,-0.441567,-0.530443,-0.899614,-0.999021,-0.368837,-0.476477,0
1067,-1.703889,-1.263858,-0.062832,-0.157307,0.185283,-0.078776,0.038057,-0.036909,-0.094399,-0.058181,...,1.680841,1.707713,1.694328,1.501562,1.430459,1.818339,1.806533,1.783857,1.649147,1


In [25]:
preproc_valid_meta_path = os.path.join(above_dir_preproc, "Xy_valid_meta_v47.csv")
preproc_valid_feats_path = os.path.join(above_dir_preproc, "Xy_valid_feats_v47.csv")

use_preproc_valid_now = (
    USE_PREPROC
    and os.path.exists(preproc_valid_meta_path)
    and os.path.exists(preproc_valid_feats_path)
)
if USE_PREPROC and (not use_preproc_valid_now):
    print(
        "Preprocessed valid files not found at:",
        above_dir_preproc,
        " -> assembling features instead.",
    )

if use_preproc_valid_now:
    Xy_valid_meta = pd.read_csv(preproc_valid_meta_path)
    Xy_valid_feats = pd.read_csv(preproc_valid_feats_path)
    Xy_valid_meta["clust_id"] = kmeans.predict(np.array(Xy_valid_meta[HBA_probs]))
    Xy_valid_feats["clust_id"] = Xy_valid_meta["clust_id"]
else:
    Xy_valid_meta = (train_meta[valid_rows_bool])[::VALID_DOWNSEL].copy()
    Xy_valid_meta = Xy_valid_meta.reset_index().drop(columns=["index"])
    print("Number of samples used for Validation =", len(Xy_valid_meta))

    Xy_valid_feats = assemble_features(
        Xy_valid_meta,
        traintest="validation",
        smooth_width=SMOOTH_WIDTH,
        SHOW_PLOT=False,
    )
    Xy_valid_meta.to_csv(
        os.path.join(above_dir_preproc, "Xy_valid_meta_v47.csv"),
        header=True,
        index=False,
        float_format="%.6f",
    )
    Xy_valid_feats.to_csv(
        os.path.join(above_dir_preproc, "Xy_valid_feats_v47.csv"),
        header=True,
        index=False,
        float_format="%.6f",
    )



Preprocessed valid files not found at: /kaggle/working/  -> assembling features instead.
Number of samples used for Validation = 186


/tmp/ipykernel_11/3100455626.py:91: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  middle4s[ibin + ioff] = middle4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:91: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  middle4s[ibin + ioff] = middle4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:92: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ratio4s[ibin + ioff] = ratio4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:92: Futu

... 100 done...


/tmp/ipykernel_11/3100455626.py:91: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  middle4s[ibin + ioff] = middle4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:91: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  middle4s[ibin + ioff] = middle4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:92: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ratio4s[ibin + ioff] = ratio4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:92: Futu

In [26]:
Xy_valid_feats



,LL_0.59,LL_0.98,LL_1.95,LL_2.93,LL_3.91,LL_4.88,LL_5.86,LL_6.84,LL_7.81,LL_8.79,...,Median,LLmean,LLmedian,RLmean,RLmedian,LPmean,LPmedian,RPmean,RPmedian,clust_id
0,-1.384369,-0.958363,-0.241366,-0.293769,-0.214329,0.198089,-0.023498,-0.001992,-0.098378,-0.147788,...,-0.032530,-0.101081,-0.157095,0.252325,0.210142,-0.068448,-0.092035,0.151848,0.110920,1
1,-1.241828,-0.937502,-0.386174,-0.103252,-0.383654,0.235917,0.267287,-0.031262,0.247092,0.085855,...,-0.167126,-0.160530,-0.237089,-0.103738,-0.115309,-0.172756,-0.241870,-0.008285,-0.032666,5
2,-0.735462,-0.126165,0.254244,0.063509,-0.122807,-0.040109,-0.069006,-0.104399,-0.012103,-0.010566,...,-0.124766,0.258933,0.246583,-0.208715,-0.217765,0.149355,0.123485,-0.341343,-0.329324,1
3,0.392368,0.382964,-0.301916,-0.826050,-0.893800,-0.963428,-0.802532,-0.677686,-0.905944,-0.897308,...,1.456447,1.165064,0.843976,1.969438,1.867744,1.179070,0.877440,1.969966,1.862084,1
4,-0.002243,0.250393,0.359189,0.433188,0.010104,-0.421552,-0.434744,-0.500485,-0.555926,-0.617792,...,-0.594472,-0.476107,-0.662615,-0.581300,-0.785358,-0.304027,-0.500694,-0.381547,-0.533268,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
181,-1.718228,-1.340963,-0.934482,-0.604090,-0.736244,-0.536415,-0.516990,0.010317,-0.040638,0.023193,...,-0.196817,-0.234453,-0.214029,-0.212530,-0.163258,-0.264608,-0.200953,-0.183148,-0.163274,2
182,-1.744319,-1.564850,-2.061693,-2.137017,-2.060589,-1.839782,-1.633328,-1.274850,-1.145987,-1.232514,...,-0.179074,-0.551125,-0.652296,1.122422,0.695252,-0.763975,-0.833314,1.328662,1.022836,0
183,-1.144014,-0.747998,-0.337173,-0.302518,-0.155544,0.032439,0.037702,0.023741,0.092118,0.131712,...,-0.134623,-0.097155,-0.079339,-0.226083,-0.433714,-0.014204,-0.007294,-0.084762,-0.302223,7
184,-0.645370,-0.540305,-0.543736,-0.355366,-0.003411,0.010243,0.264239,0.032439,-0.259814,-0.228291,...,-1.058081,-1.024029,-1.106108,-0.991064,-1.114028,-0.945943,-1.104188,-0.646893,-0.804187,0


In [27]:
X = Xy_train_feats.drop(columns=["clust_id"])
y = Xy_train_feats.clust_id

Xlr = X.drop(columns=X.columns[-10:])

lr1_feature_names_ = None
lr2_feature_names_ = None

if USE_LR1:
    Xlr1 = Xlr.iloc[:, 0 : int(len(Xlr.columns) / 2)]
    lr1_feature_names_ = Xlr1.columns.to_list()
    lrmodel1 = LogisticRegression(
        penalty="l1",
        C=LR1_C,
        solver="saga",
        max_iter=1500,
        multi_class="multinomial",
        n_jobs=-1,
        random_state=GLOBAL_SEED,
    ).fit(Xlr1, y)

    if SHOW_EDA_PLOTS:
        plt.figure(figsize=(8, 4))
        plt.plot(lrmodel1.coef_.T, "-", alpha=0.5)
        plt.plot(lrmodel1.coef_.T, ".", alpha=1.0)
        plt.title("Logistic Regression coefficients (colored by cluster)")
        plt.show()

    print("\nLR1 model score for X,y = {:.1f}%\n".format(100 * lrmodel1.score(Xlr1, y)))

if USE_LR2:
    Xlr2 = Xlr.iloc[:, int(len(Xlr.columns) / 2) :]
    lr2_feature_names_ = Xlr2.columns.to_list()
    lrmodel2 = LogisticRegression(
        penalty="l1",
        C=LR2_C,
        solver="saga",
        max_iter=1500,
        multi_class="multinomial",
        n_jobs=-1,
        random_state=GLOBAL_SEED,
    ).fit(Xlr2, y)

    if SHOW_EDA_PLOTS:
        plt.figure(figsize=(8, 4))
        plt.plot(lrmodel2.coef_.T, "-", alpha=0.5)
        plt.plot(lrmodel2.coef_.T, ".", alpha=1.0)
        plt.title("Logistic Regression coefficients (colored by cluster)")
        plt.show()

    print("\nLR2 model score for X,y = {:.1f}%\n".format(100 * lrmodel2.score(Xlr2, y)))




LR1 model score for X,y = 34.2%



In [28]:
Xy_train_wLRfeats = Xy_train_feats.copy()
X = Xy_train_feats.drop(columns=["clust_id"])
Xlr = X.drop(columns=X.columns[-10:])

if USE_LR1:
    Xlr1 = Xlr.iloc[:, 0 : int(len(Xlr.columns) / 2)]
    Xlr1 = align_features_for_lr(Xlr1, lr1_feature_names_)
    lrprobas = lrmodel1.predict_proba(Xlr1)
    for iadd in range(NUM_CLUSTS):
        Xy_train_wLRfeats["lr" + str(iadd)] = lrprobas[:, iadd] + LR_BLUR * (
            np.random.rand(len(lrprobas)) - 0.5
        )
    if SHOW_EDA_PLOTS:
        plt.figure(figsize=(7, 2.5))
        plt.hist(np.max(lrmodel1.predict_proba(Xlr1), axis=1), bins=50)
        plt.xlim(-0.5 * LR_BLUR, 1.01)
        plt.title("Training max LR1 proba values (pre-blur)")
        plt.show()

if USE_LR2:
    Xlr2 = Xlr.iloc[:, int(len(Xlr.columns) / 2) :]
    Xlr2 = align_features_for_lr(Xlr2, lr2_feature_names_)
    lrprobas = lrmodel2.predict_proba(Xlr2)
    for iadd in range(NUM_CLUSTS):
        Xy_train_wLRfeats["rlr" + str(iadd)] = lrprobas[:, iadd] + LR_BLUR * (
            np.random.rand(len(lrprobas)) - 0.5
        )
    if SHOW_EDA_PLOTS:
        plt.figure(figsize=(7, 2.5))
        plt.hist(np.max(lrmodel2.predict_proba(Xlr2), axis=1), bins=50)
        plt.xlim(-0.5 * LR_BLUR, 1.01)
        plt.title("Training max LR2 proba values (pre-blur)")
        plt.show()

Xy_valid_wLRfeats = Xy_valid_feats.copy()
X = Xy_valid_feats.drop(columns=["clust_id"])
Xlr = X.drop(columns=X.columns[-10:])

if USE_LR1:
    Xlr1 = Xlr.iloc[:, 0 : int(len(Xlr.columns) / 2)]
    Xlr1 = align_features_for_lr(Xlr1, lr1_feature_names_)
    lrprobas = lrmodel1.predict_proba(Xlr1)
    for iadd in range(NUM_CLUSTS):
        Xy_valid_wLRfeats["lr" + str(iadd)] = lrprobas[:, iadd]
    if SHOW_EDA_PLOTS:
        plt.figure(figsize=(7, 2.5))
        plt.hist(np.max(lrmodel1.predict_proba(Xlr1), axis=1), bins=50)
        plt.xlim(-0.5 * LR_BLUR, 1.01)
        plt.title("Validation max LR1 proba values (pre-blur)")
        plt.show()

if USE_LR2:
    Xlr2 = Xlr.iloc[:, int(len(Xlr.columns) / 2) :]
    Xlr2 = align_features_for_lr(Xlr2, lr2_feature_names_)
    lrprobas = lrmodel2.predict_proba(Xlr2)
    for iadd in range(NUM_CLUSTS):
        Xy_valid_wLRfeats["rlr" + str(iadd)] = lrprobas[:, iadd]
    if SHOW_EDA_PLOTS:
        plt.figure(figsize=(7, 2.5))
        plt.hist(np.max(lrmodel2.predict_proba(Xlr2), axis=1), bins=50)
        plt.xlim(-0.5 * LR_BLUR, 1.01)
        plt.title("Validation max LR2 proba values (pre-blur)")
        plt.show()



In [29]:
# Score/stability: ensure RF sees consistent, aligned features for fit() even if cached CSVs differ
X_for_rf_train = Xy_train_wLRfeats.drop(columns=["clust_id"])
y = Xy_train_wLRfeats.clust_id

ave_oob = []
nfits = 3
for ifit in range(nfits):
    rfmodel = RandomForestClassifier(
        n_estimators=100,
        min_samples_leaf=9,
        max_features=0.20,
        max_samples=0.8,
        oob_score=True,
        class_weight="balanced_subsample",
        n_jobs=-1,
        verbose=0,
        random_state=GLOBAL_SEED + ifit,
    ).fit(X_for_rf_train, y)
    ave_oob.append(rfmodel.oob_score_)

if SHOW_EDA_PLOTS:
    sort_inds = rfmodel.feature_importances_.argsort()
    plt.figure(figsize=(4, 7))
    plt.barh(
        rfmodel.feature_names_in_[sort_inds], rfmodel.feature_importances_[sort_inds]
    )
    plt.ylim(len(sort_inds) - 40, len(sort_inds) + 0.2)
    plt.title("Feature Importances (top 40)")
    plt.show()

print(
    "\nRF model ave OOB score = {:.1f}% +/- {:.1f}".format(
        100 * np.mean(ave_oob), 100 * np.std(ave_oob)
    )
)
print(
    "\nRF model score for X,y = {:.1f}%\n".format(
        100 * rfmodel.score(X_for_rf_train, y)
    )
)




RF model ave OOB score = 35.2% +/- 1.5

RF model score for X,y = 79.4%



In [30]:
# Metric-aligned improvement for KL: tune taming on TRAIN using soft RF memberships (predict_proba)
X_train_for_rf = align_features_for_model(
    Xy_train_wLRfeats.drop(columns=["clust_id"]), rfmodel.feature_names_in_
)

train_membership = rfmodel.predict_proba(X_train_for_rf)
train_center_probs = centers_from_membership(train_membership, clust_centers)

# Convert soft center probs -> pseudo pred_ids by argmax ONLY for taming search routine,
# but compute KL with soft probabilities afterwards (lower KL expected).
Xy_train_meta["pred_id"] = np.argmax(train_center_probs, axis=1)

solution_probs = Xy_train_meta[HBA_probs].copy()
solution_probs.columns = HBA_votes
pred_ids = Xy_train_meta["pred_id"].to_numpy()

best_fracs_train, best_centers_train = find_best_tamed_kl(
    solution_probs=solution_probs, pred_ids=pred_ids, centers=clust_centers
)

# Use soft membership with tamed centers for the actual KL calculation (closer to metric)
submission_probs = solution_probs.copy()
soft_tamed_probs = centers_from_membership(train_membership, best_centers_train)
for iprob, col in enumerate(HBA_votes):
    submission_probs[col] = soft_tamed_probs[:, iprob]

this_kl = kld_score(solution_probs, submission_probs)
print(
    "TRAIN taming fractions:\n",
    best_fracs_train,
    "\nand centers:\n",
    best_centers_train,
)
print("\nTRAIN KL from soft-membership + tamed centers: {:.4f}".format(this_kl))



TRAIN taming fractions:
 [0.03 0.03 0.03 0.33 0.03 0.03 0.03 0.03 0.03] 
and centers:
 [[0.202544 0.129    0.125214 0.135762 0.174718 0.23276 ]
 [0.231243 0.128329 0.124826 0.134809 0.173948 0.206846]
 [0.202144 0.128268 0.124836 0.135415 0.202557 0.206781]
 [0.155778 0.246995 0.103325 0.145851 0.126993 0.221058]
 [0.202282 0.128887 0.1248   0.161455 0.175759 0.206818]
 [0.204319 0.129546 0.147259 0.134876 0.174682 0.209318]
 [0.202975 0.129711 0.124988 0.150679 0.175238 0.216409]
 [0.202835 0.155688 0.125089 0.135093 0.173977 0.207317]
 [0.202469 0.128978 0.126986 0.136648 0.187077 0.217842]]

TRAIN KL from soft-membership + tamed centers: 1.3421


In [31]:
# Validation KL (use soft membership; no change to split/core training logic)
X_valid_for_rf = align_features_for_model(
    Xy_valid_wLRfeats.drop(columns=["clust_id"]), rfmodel.feature_names_in_
)

valid_membership = rfmodel.predict_proba(X_valid_for_rf)
soft_valid_probs = centers_from_membership(valid_membership, best_centers_train)

solution_probs = Xy_valid_meta[HBA_probs].copy()
solution_probs.columns = HBA_votes
submission_probs = solution_probs.copy()
for iprob, col in enumerate(HBA_votes):
    submission_probs[col] = soft_valid_probs[:, iprob]

this_kl = kld_score(solution_probs, submission_probs)
print(
    "\nVALID KL using soft-membership + TRAIN-tuned tamed centers: {:.4f}".format(
        this_kl
    )
)




VALID KL using soft-membership + TRAIN-tuned tamed centers: 1.3503


In [32]:
best_centers_for_test = best_centers_train

mean_all_probs = train_meta[HBA_probs].mean(axis=0).to_numpy(dtype=float)
mean_all_probs = np.clip(mean_all_probs, 1e-6, 1.0)
mean_all_probs = mean_all_probs / mean_all_probs.sum()

sample_sub = pd.read_csv(above_dir + "sample_submission.csv")
sample_sub["eeg_id"] = sample_sub["eeg_id"].astype(np.int64)

test_meta_local = test_meta.copy()
test_meta_local["eeg_id"] = test_meta_local["eeg_id"].astype(np.int64)

test_meta_unique = test_meta_local.drop_duplicates(
    subset=["eeg_id"], keep="first"
).reset_index(drop=True)
print("Test meta rows:", len(test_meta_local), " Unique eeg_id:", len(test_meta_unique))

preproc_test_feats_path = os.path.join(above_dir_preproc, "X_test_feats_v47.csv")
use_preproc_test_now = USE_PREPROC and os.path.exists(preproc_test_feats_path)

if use_preproc_test_now:
    Xy_test_feats = pd.read_csv(preproc_test_feats_path)
    print(
        "Loaded cached test features:",
        preproc_test_feats_path,
        "shape=",
        Xy_test_feats.shape,
    )
    if len(Xy_test_feats) != len(test_meta_unique):
        print(
            "Cached test features rowcount mismatch (cached={}, unique_meta={}); rebuilding features.".format(
                len(Xy_test_feats), len(test_meta_unique)
            )
        )
        Xy_test_feats = assemble_features(
            test_meta_unique,
            traintest="test",
            smooth_width=SMOOTH_WIDTH,
            SHOW_PLOT=False,
        )
        Xy_test_feats.to_csv(
            preproc_test_feats_path,
            header=True,
            index=False,
            float_format="%.6f",
        )
else:
    Xy_test_feats = assemble_features(
        test_meta_unique, traintest="test", smooth_width=SMOOTH_WIDTH, SHOW_PLOT=False
    )
    Xy_test_feats.to_csv(
        preproc_test_feats_path,
        header=True,
        index=False,
        float_format="%.6f",
    )
    print(
        "Cached test features to:",
        preproc_test_feats_path,
        "shape=",
        Xy_test_feats.shape,
    )

Xy_test_wLRfeats = Xy_test_feats.copy()
Xlr = Xy_test_feats.drop(columns=Xy_test_feats.columns[-10:])

if USE_LR1:
    Xlr1 = Xlr.iloc[:, 0 : int(len(Xlr.columns) / 2)]
    Xlr1 = align_features_for_lr(Xlr1, lr1_feature_names_)
    lrprobas = lrmodel1.predict_proba(Xlr1)
    for iadd in range(NUM_CLUSTS):
        Xy_test_wLRfeats["lr" + str(iadd)] = lrprobas[:, iadd]
if USE_LR2:
    Xlr2 = Xlr.iloc[:, int(len(Xlr.columns) / 2) :]
    Xlr2 = align_features_for_lr(Xlr2, lr2_feature_names_)
    lrprobas = lrmodel2.predict_proba(Xlr2)
    for iadd in range(NUM_CLUSTS):
        Xy_test_wLRfeats["rlr" + str(iadd)] = lrprobas[:, iadd]

# Metric-aligned improvement for KL: use soft membership for test predictions
X_test_for_rf = align_features_for_model(
    Xy_test_wLRfeats,
    (
        rfmodel.feature_names_in_
        if hasattr(rfmodel, "feature_names_in_")
        else Xy_test_wLRfeats.columns
    ),
)
test_membership = rfmodel.predict_proba(X_test_for_rf)
soft_test_probs = centers_from_membership(test_membership, best_centers_for_test)

test_pred = test_meta_unique[["eeg_id"]].copy()
for i, col in enumerate(HBA_votes):
    test_pred[col] = soft_test_probs[:, i]

test_pred = normalize_submission_probas(test_pred, HBA_votes, eps=1e-6)

test_submit = sample_sub[["eeg_id"]].merge(test_pred, on="eeg_id", how="left")

for col in HBA_votes:
    if col not in test_submit.columns:
        test_submit[col] = 1.0 / HBA_number
test_submit[HBA_votes] = test_submit[HBA_votes].fillna(1.0 / HBA_number)

test_submit = normalize_submission_probas(test_submit, HBA_votes, eps=1e-6)

row_sums = test_submit[HBA_votes].sum(axis=1).to_numpy()
print(
    "Submission row_sums: min/mean/max =",
    float(np.min(row_sums)),
    float(np.mean(row_sums)),
    float(np.max(row_sums)),
)

assert len(test_submit) == len(sample_sub), "Row count mismatch vs sample_submission."
assert list(test_submit.columns) == list(
    sample_sub.columns
), "Submission columns mismatch."
assert np.isfinite(
    test_submit[HBA_votes].to_numpy()
).all(), "Non-finite probabilities in submission."
assert np.allclose(row_sums, 1.0, atol=1e-5), "Row probabilities do not sum to 1."
assert test_submit["eeg_id"].notna().all(), "Missing eeg_id values in submission."

print(test_submit.head())

test_submit.to_csv(
    "submission.csv", header=True, index=False, na_rep="", float_format="%.6f"
)
print("Wrote submission.csv with shape:", test_submit.shape)

Test meta rows: 9850  Unique eeg_id: 1693


/tmp/ipykernel_11/3100455626.py:91: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  middle4s[ibin + ioff] = middle4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:91: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  middle4s[ibin + ioff] = middle4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:92: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ratio4s[ibin + ioff] = ratio4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:92: Futu

... 100 done...


/tmp/ipykernel_11/3100455626.py:91: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  middle4s[ibin + ioff] = middle4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:91: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  middle4s[ibin + ioff] = middle4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:92: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ratio4s[ibin + ioff] = ratio4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:92: Futu

... 200 done...


...

... 1500 done...


/tmp/ipykernel_11/3100455626.py:91: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  middle4s[ibin + ioff] = middle4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:91: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  middle4s[ibin + ioff] = middle4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:92: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ratio4s[ibin + ioff] = ratio4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:92: Futu

... 1600 done...


/tmp/ipykernel_11/3100455626.py:91: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  middle4s[ibin + ioff] = middle4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:91: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  middle4s[ibin + ioff] = middle4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:92: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ratio4s[ibin + ioff] = ratio4spre[ibin + ioff]
/tmp/ipykernel_11/3100455626.py:92: Futu

Cached test features to: /kaggle/working/X_test_feats_v47.csv shape= (1693, 178)
Submission row_sums: min/mean/max = 0.9999999999999998 1.0 1.0000000000000002
       eeg_id  seizure_vote  lpd_vote  gpd_vote  lrda_vote  grda_vote  \
0  2578018731      0.201231  0.144664  0.126340   0.140572   0.173015   
1  2578018731      0.201231  0.144664  0.126340   0.140572   0.173015   
2  2578018731      0.201231  0.144664  0.126340   0.140572   0.173015   
3  2578018731      0.201231  0.144664  0.126340   0.140572   0.173015   
4  1706196508      0.201589  0.143524  0.125171   0.139447   0.174473   

   other_vote  
0    0.214177  
1    0.214177  
2    0.214177  
3    0.214177  
4    0.215796  
Wrote submission.csv with shape: (9850, 7)
